# EDGAR - The SEC's Electronic Data Gathering, Analysis, and Retrieval (EDGAR) system

There are a lot of EDGAR-related endpoints at https://www.sec.gov.  Most of the EDGAR overview can be found through [the link to search public filings](https://www.sec.gov/search-filings).  The EDGAR search itself is available at https://www.sec.gov/edgar/search/.  There is a good amount of additional info in the [FAQ](https://www.sec.gov/edgar/search/efts-faq.html), the [API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces), and the [data resources](https://www.sec.gov/data-research/sec-data-resources).

### Open government data in general

Slightly off-topic here, but there are a lot of additional open datasets at https://data.gov/open-gov/.

## Imports

In [1]:
import requests
import json
from io import StringIO
import pandas as pd

## Headers

According to the [developer FAQ](https://www.sec.gov/about/webmaster-frequently-asked-questions#developers), you must declare your identity in the `User-Agent` header.

In [2]:
with open('../../../../../.config/edgar/user-agent', 'r') as file:
    user_agent = file.read().strip()
headers = {
    'User-Agent': user_agent
}

## CIK data

EDGAR's Central Index Key (CIK) is the uuid used to track filing registered by a particular entity.  There is a large list of these downloadable at https://www.sec.gov/Archives/edgar/cik-lookup-data.txt .

There are alernative (possibly old/out-dated) mappings by ticker https://www.sec.gov/files/company_tickers.json and by ticket and exchange https://www.sec.gov/files/company_tickers_exchange.json .

### All CIK data

This appears to be--at best--space-separated values, connecting CIKs and entity names.  I checked that it contains Enron (several, actually...), so it appears to have a historical record beyond actively registered entities.

Systematically sifting through this is left for future work.

### CIK by ticker

In [3]:
cik_ticker = pd.read_json('../../data/edgar/company_tickers.json')
cik_ticker = cik_ticker.transpose()
cik_ticker

,cik_str,ticker,title
0,1045810,NVDA,NVIDIA CORP
1,320193,AAPL,Apple Inc.
2,1652044,GOOGL,Alphabet Inc.
3,789019,MSFT,MICROSOFT CORP
4,1018724,AMZN,AMAZON COM INC
...,...,...,...
10420,2086264,VHCPU,Vine Hill Capital Investment Corp. II
10421,2097364,TMTSU,Spartacus Acquisition Corp. II
10422,2107960,SWCPF,SWCC Corp/ADR
10423,2093654,ZJNGF,Zijin Gold International Co Limited/ADR


#### Test of $MSFT CIK lookup

In [4]:
msft_cik_ticker = cik_ticker[cik_ticker['ticker'] == 'MSFT']
print(msft_cik_ticker)
print("Extracted MSFT CIK: ", msft_cik_ticker['cik_str'].iloc[0])

  cik_str ticker           title
3  789019   MSFT  MICROSOFT CORP
Extracted MSFT CIK:  789019


#### Test of $ENE CIK lookup

It looks like this dataset does not contain Enron.  I suspect this may be missing companies that are no longer registered with the SEC.

In [31]:
# Check ticker
cik_ticker[cik_ticker['ticker'] == 'ENE']

,cik_str,ticker,title


In [ ]:
# Check title strings
cik_ticker[cik_ticker['title'].str.contains("enron", case=False)]

,cik_str,ticker,title


### CIK by ticker _and_ exchange

#### Reformat the JSON to the form expected by pandas.read_json

In [5]:
def create_cik_exchange_json(filepath: str) -> str:
    """
    Formats the input company_tickers_exchange.json file to the form expected by pandas.read_json().
    """
    with open(filepath, 'r') as file:
        text = file.read().strip()
    cik_exchange = json.loads(text)
    assert validate_cik_exchange_data(cik_exchange['data'])
    return convert_cik_exchange_data_to_json(cik_exchange)

def validate_cik_exchange_data(data: dict, length: int=4) -> bool:
    """
    Validates that the input array of items in the CIK breakdown by ticker and exchange.
    """
    for item in data:
        if len(item) != length:
            print("Invalid length: ", item)
            return False
    return True

def convert_cik_exchange_data_to_json(unformatted_data: dict) -> str:
    """
    Converts the dict-ified version of the company_tickers_exchange.json data into a JSON string compatible with pandas.read_json().
    """
    fields = unformatted_data['fields']
    formatted = {}
    index = 0
    for item in unformatted_data['data']:
        formatted_item = {}
        for column in range(len(fields)):
            formatted_item[fields[column]] = item[column]
        formatted[index] = formatted_item
        index += 1
    return json.dumps(formatted)

#### Read in the CIK data labeled by ticker _and_ exchange

In [6]:
cik_ticker_exchange_json = create_cik_exchange_json('../../data/edgar/company_tickers_exchange.json')
cik_ticker_exchange = pd.read_json(StringIO(cik_ticker_exchange_json))  # raw string is deprecated
cik_ticker_exchange = cik_ticker_exchange.transpose()
cik_ticker_exchange


,cik,name,ticker,exchange
0,1045810,NVIDIA CORP,NVDA,Nasdaq
1,320193,Apple Inc.,AAPL,Nasdaq
2,1652044,Alphabet Inc.,GOOGL,Nasdaq
3,789019,MICROSOFT CORP,MSFT,Nasdaq
4,1018724,AMAZON COM INC,AMZN,Nasdaq
...,...,...,...,...
10407,2108124,"Odakyu Electric Railway Co., Ltd./ADR",ODERF,OTC
10408,2108180,NOF Corporation/ADR,NOFCF,OTC
10409,2108185,Azbil Corporation/ADR,YMATF,OTC
10410,2109234,BELIMO Holding AG/ADR,BLHWF,OTC


#### Test of $MSFT CIK lookup

In [7]:
msft_cik_ticker_exchange = cik_ticker_exchange[cik_ticker_exchange['ticker'] == 'MSFT']
print(msft_cik_ticker_exchange)
print("Extracted CIK: ", msft_cik_ticker_exchange['cik'].iloc[0])

      cik            name ticker exchange
3  789019  MICROSOFT CORP   MSFT   Nasdaq
Extracted CIK:  789019


#### Test of $ENE CIK lookup

It looks like this data is also missing entities that are no longer registered with the SEC.  

In [32]:
# Check ticker
cik_ticker_exchange[cik_ticker_exchange['ticker'] == 'ENE']

,cik,name,ticker,exchange


In [39]:
# Check title strings
cik_ticker_exchange[cik_ticker_exchange['name'].str.contains("enron", case=False)]

,cik,name,ticker,exchange


## Company filing data

There are several endpoints, data dumps, directories, and search tools available.

The relevant API endpoints seem to be:
- `https://data.sec.gov/api/xbrl/companyconcept/CIK<cik_str>/<us-gaap|dei>/<company_concept_key>.json`
- `https://data.sec.gov/api/xbrl/companyfacts/CIK<cik_str>.json`

There are additional references to bulk zip files, presumably with similar underlying data (particularly for the xbrl/companyfacts.zip link):
- https://www.sec.gov/Archives/edgar/daily-index/xbrl/companyfacts.zip
- https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip

The EDGAR search can be accessed at https://www.sec.gov/edgar/search/# and allows for (manual) searching on `CIK`.  Additional fields like the start/end dates and text search can be leveraged to try and isolate particular filings.  I only saw data back to the turn of the century (2000, with some replicated 1999 data in the form of YoY comparisons) through these searches, but maybe I was using it incorrectly.

Finally, the oldest filings seem to be most easily found directly from the `www.sec.gov/Archives/edgar/data/<CIK>` directory structure.  I found [data filed in 1994](https://www.sec.gov/Archives/edgar/data/789019/000095010994000252/0000950109-94-000252.txt), which covered the the period ending December 31st, 1993.  This even included some 1992 data in YoY comparisons.  Curiously, the directory _does not include leading 0s_ in the `CIK`.  For example, the path for $MSFT data is at https://www.sec.gov/Archives/edgar/data/789019/.  It's noted in [a page on accessing EDGAR data](https://www.sec.gov/search-filings/edgar-search-assistance/accessing-edgar-data) that each `CIK` directory (and its subdirectories) provide files to assist automated crawling of these directories for documents.  Traversal through `www.sec.gov/Archives/edgar/data/` without a provided `CIK`, however, is _not_ allowed.

### Padding CIKs

CIKs are meant to be 10-digit unique IDs.  Nevertheless, EDGAR is somewhat inconsistent on whether the leading 0s are needed for a particular path, and these leading 0s are not provided in EDGAR's provided ticker<->CIK mapping.  The function(s) below pad the front of an input CIK with 0s for convenience when required.

In [8]:
def pad_cik(cik: str | int) -> str:
    """
    Adds leading 0s to a CIK.
    """
    cik = str(cik)
    if len(cik) < 10:
        padding = 10 - len(cik)
        cik = ('0' * padding) + cik
    return cik

### $MSFT filings using `api/xrbl/companyfacts/` endpoint

In [9]:
msft_cik = msft_cik_ticker_exchange['cik'].iloc[0]
msft_cik = pad_cik(msft_cik)
msft_company_facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{msft_cik}.json"

company_facts_response = requests.get(msft_company_facts_url, headers=headers)
company_facts_response.status_code

200

#### Share history

I wanted to see how fine-grained the data here was, and how far back it went.  There are several keys related to shares outstanding and share repurchases, but they seem to only go back to 2007.  The filing dates appear to start in 2010, which tracks more or less with a statement in the section on XBRL Data APIs of [the API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces).  In particular, it sounds like including these convenient, searchable tags was only enforced after 2009.  Previous years are available when searching through the directory structure, but the documents may simply be searchable .txt files.

In [10]:
msft_facts = company_facts_response.json()

# Interesting keys:
# facts: {dei: {}, us-gaap: {}}
# dei: {EntityCommonStockSharesOutstanding}
# us-gaap: {StockRepurchasedDuringPeriodShares}
# us-gaap: {CommonStockSharesOutstanding}
# us-gaap: {WeightedAverageNumberOfSharesOutstandingBasic}
# us-gaap: {WeightedAverageNumberOfDilutedSharesOutstanding}
# us-gaap: {PaymentsForRepurchaseOfCommonStock}
# us-gaap: {StockRepurchasedDuringPeriodShares}
# us-gaap: {StockRepurchasedDuringPeriodValue}

first_entry = msft_facts['facts']['us-gaap']['CommonStockSharesOutstanding']['units']['shares'][0]
print("Values in the first entry:")
for key in first_entry:
    print(f"{key}: {first_entry[key]}")

Values in the first entry:
end: 2007-06-30
val: 9380000000
accn: 0001193125-10-171791
fy: 2010
fp: FY
form: 10-K
filed: 2010-07-30
frame: CY2007Q2I


### $MSFT filings using `api/xbrl/companyconcept/` endpoint

In [11]:
msft_cik = msft_cik_ticker_exchange['cik'].iloc[0]
msft_cik = pad_cik(msft_cik)
msft_company_concept_url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{msft_cik}/us-gaap/CommonStockSharesOutstanding.json"

company_concepts_response = requests.get(msft_company_concept_url, headers=headers)
company_concepts_response.status_code

200

#### Share history

It seems like the two API xrbl endpoints are using the same underlying data, which starts in 2007 despite EDGAR containing filings back to at least FY 2001.

In [12]:
msft_concepts = company_concepts_response.json()

first_entry = msft_concepts['units']['shares'][0]
print("Values in the first entry:")
for key in first_entry:
    print(f"{key}: {first_entry[key]}")

Values in the first entry:
end: 2007-06-30
val: 9380000000
accn: 0001193125-10-171791
fy: 2010
fp: FY
form: 10-K
filed: 2010-07-30
frame: CY2007Q2I


### $MSFT filings using the `submissions/` endpoint

In [13]:
msft_cik = msft_cik_ticker_exchange['cik'].iloc[0]
msft_cik = pad_cik(msft_cik)
submissions_url = f"https://data.sec.gov/submissions/CIK{msft_cik}.json"
submissions_response = requests.get(submissions_url, headers=headers)
submissions_response.status_code

200

#### Quick look at $MSFT filings (1 of 3 JSON filing histories)

It seems like this might be useful as a way to derive accession numbers for filings based on type and date of filing.  But there's no actual data from the filings here.

In [14]:
top_level = submissions_response.json()

print("Additional submission history:")
for file in top_level['filings']['files']:
    print(file)

Additional submission history:
{'name': 'CIK0000789019-submissions-001.json', 'filingCount': 2002, 'filingFrom': '2008-02-04', 'filingTo': '2019-11-30'}
{'name': 'CIK0000789019-submissions-002.json', 'filingCount': 1414, 'filingFrom': '1994-02-14', 'filingTo': '2008-02-02'}


#### Looking at the oldest $MSFT submissions

You can access the raw data for a filing from the `accessionNumber` field, with the hyphens.  For example, the oldest 10-Q filing for $MSFT is available at https://www.sec.gov/Archives/edgar/data/789019/0000950109-94-000252.txt.

It's not clear to me if this pattern is stable year-to-year as document types changed.  For example, this oldest 10-Q is missing a `primaryDocument` entry, so maybe the default is at `/<accessionNumber>.txt` unless otherwise stated.  Although it also seems like the raw text dump of filing (with html formatting) is _always_ at `/<accessionNumber>.txt`.

In [15]:
msft_cik = msft_cik_ticker_exchange['cik'].iloc[0]
msft_cik = pad_cik(msft_cik)
oldest_submissions_url = f"https://data.sec.gov/submissions/CIK{msft_cik}-submissions-002.json"
oldest_response = requests.get(oldest_submissions_url, headers=headers)
oldest_response.status_code

200

In [16]:
oldest_submissions = oldest_response.json()
for key in oldest_submissions:
    print(f"{key}: {oldest_submissions[key][0]}")

accessionNumber: 0000950123-08-001050
filingDate: 2008-02-01
reportDate: 
acceptanceDateTime: 2008-02-01T21:58:33.000Z
act: 
form: 425
fileNumber: 
filmNumber: 
items: 
core_type: 425
size: 187400
isXBRL: 0
isInlineXBRL: 0
primaryDocument: y47867ce425.htm
primaryDocDescription: 425


### Confirming that the `submissions/` endpoint contains $ENE (Enron) data

Note that the `submissions/` endpoint wants the padded `CIK`, but the actual filings are located in directories with the leading 0s removed.

In [40]:
ene_cik = '0000072859'
ene_url = f"https://data.sec.gov/submissions/CIK{ene_cik}.json"
ene_response = requests.get(ene_url, headers=headers)
ene_response.status_code

200

In [51]:
ene_submissions = ene_response.json()

for key in ene_submissions:
    if 'filings' in key.lower():
        print(f"{key}: {len(ene_submissions[key]['recent']['accessionNumber'])} not shown")
        continue  # Skip the huge filings data payload
    print(f"{key}: {ene_submissions[key]}")

cik: 0000072859
entityType: operating
sic: 6211
sicDescription: Security Brokers, Dealers & Flotation Companies
ownerOrg: None
insiderTransactionForOwnerExists: 0
insiderTransactionForIssuerExists: 0
name: ENRON CORP
tickers: []
exchanges: []
ein: 470255140
lei: None
description: 
website: 
investorWebsite: 
category: 
fiscalYearEnd: 1231
stateOfIncorporation: DE
stateOfIncorporationDescription: DE
addresses: {'mailing': {'street1': 'PO BOX 1188', 'street2': None, 'city': 'HOUSTON', 'stateOrCountry': 'TX', 'zipCode': '77002', 'stateOrCountryDescription': 'TX', 'isForeignLocation': None, 'foreignStateTerritory': None, 'country': None, 'countryCode': None}, 'business': {'street1': '1400 SMITH ST', 'street2': None, 'city': 'HOUSTON', 'stateOrCountry': 'TX', 'zipCode': '77002', 'stateOrCountryDescription': 'TX', 'isForeignLocation': None, 'foreignStateTerritory': None, 'country': None, 'countryCode': None}}
phone: 7138536161
flags: 
formerNames: []
filings: 93 not shown


#### Finding an old Enron 8-K

The 8-K indicated below can be found at the suggested link: https://www.sec.gov/Archives/edgar/data/72859/0000950129-94-000063.txt.  It seems that the `submissions/` endpoint can be used to find entity data, whether or not they remain registered with the SEC.

Note that the above link has the leading 0s dropped from the padded `CIK`.

In [59]:
filings = ene_submissions['filings']['recent']
for key in filings:
    print(f"{key}: {filings[key][-2]}")

accessionNumber: 0000950129-94-000063
filingDate: 1994-02-04
reportDate: 1994-02-03
acceptanceDateTime: 1994-02-04T00:00:00.000Z
act: 
form: 8-K
fileNumber: 001-03423
filmNumber: 94504441
items: 5,7
core_type: 8-K
size: 21249
isXBRL: 0
isInlineXBRL: 0
primaryDocument: 
primaryDocDescription: ENRON CORPORATION FORM 8-K


### The zip files

There are bulk, zipped data payloads at:
- https://www.sec.gov/Archives/edgar/daily-index/xbrl/companyfacts.zip
- https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip

I'm gonna go out on a limb and assume that these don't contain additional data (i.e., `us-gaap` JSON data from 1994-2007), but represent the snapshot of data reflected in the API at the time of compilation.

That said, the bulk submissions.zip may be helpful when sourcing historical filings.

## Todo/future work

- Figure out how to systematically extract data from the old, non-XBRL files.  Need to see if the .txt file structure is consistent or changes year-to-year.